In [1]:
import polars as pl
from pathlib import Path
import xlsxwriter
import os

# Cấu hình hiển thị
pl.Config.set_tbl_cols(50)

polars.config.Config

In [2]:
def get_base_path():
    paths = [r"C:/Users/huuchinh.nguyen", r"C:/Users/ADMIN"]
    for p in paths:
        if os.path.exists(p): return p
    raise FileNotFoundError("Could not find a valid user directory.")

def process_and_export_final(folder_target_name: str, output_filename: str):
    try:
        # 1. Setup Paths
        base_path = get_base_path()
        root_bi_task = f"{base_path}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task"
        folder_path = Path(f"{root_bi_task}/RAW/{folder_target_name}")
        output_path = folder_path / output_filename
        
        path_global_hc = f"{root_bi_task}/Global Report/Global HC.xlsx" 
        path_mapping = f"{root_bi_task}/Global Report/LOB Mapping Aligned OneReport/AQG_summarized.xlsx"

        print(f"Base Path: {base_path}")

        # --- CẤU HÌNH: 5 CATEGORY MỤC TIÊU ---
        TARGET_5_CATEGORIES = [
            "d_happy_response",
            "d_surprised_response", 
            "u_response",
            "e_response",
            "t_response"
        ]

        # 2. Define Target Columns
        TARGET_COLUMNS = [
            "Joined date", "Agent People Id", "Conversation Conversation Id", 
            "Agent Email ID", "Agent Queue Group Name", "Latest VA Intent", 
            "Conversation Latest VA Product", "Conversation Business Segment Name", 
            "Conversation Partner Name", "Handle (Count)", "Handle Time (Sum)", 
            "Response Count", "Response Time", "Hold Time (Sum)", 
            "Wrap Up Time (Sum)", "Talk Time (Sum)", "LOB", "Outbound", 
            "FollowUp", "Survey", "Promoter", "Detractor", "Neutral", 
            "Name", "Supervisor", "Location", "AON Status"
        ]

        # 3. Load Mapping & HC Files
        print("Loading Mapping, Magdoom HC & Global HC files...")
        try:
            df_mapping = pl.read_excel(path_mapping, sheet_name="queue_name_mapping").unique(subset=["Agent Queue Group Name"])
            
            df_global_hc = pl.read_excel(path_global_hc).select(["SSO ID", "Name", "Supervisor", "Location", "AON Status"])
            df_global_hc = df_global_hc.with_columns(pl.col("SSO ID").cast(pl.Utf8, strict=False).str.strip_chars().str.to_lowercase()).unique(subset=["SSO ID"]).rename({"Name": "Name_gl", "Supervisor": "Supervisor_gl", "Location": "Location_gl", "AON Status": "AON Status_gl"})

            df_magdoom_hc_raw = pl.read_excel(path_mapping, sheet_name="magdoom_hc")
            if "Agent Email ID" in df_magdoom_hc_raw.columns: df_magdoom_hc_raw = df_magdoom_hc_raw.rename({"Agent Email ID": "SSO ID"})
            
            df_magdoom_hc = df_magdoom_hc_raw.select(["SSO ID", "Name", "Supervisor", "Location", "AON Status"])
            df_magdoom_hc = df_magdoom_hc.with_columns(pl.col("SSO ID").cast(pl.Utf8, strict=False).str.strip_chars().str.to_lowercase()).unique(subset=["SSO ID"]).rename({"Name": "Name_md", "Supervisor": "Supervisor_md", "Location": "Location_md", "AON Status": "AON Status_md"})
            
        except Exception as e:
            print(f"Error loading mapping/HC files: {e}")
            return

        # 4. Identify Data Files
        all_files = [f for f in folder_path.glob("*") if f.name != output_filename and not f.name.startswith("~$")]
        new_look_files = [f for f in all_files if "New_Look_Excel_data" in f.name and f.suffix.lower() == ".csv"]
        oa_files = [f for f in all_files if f.name.startswith("OA_Current_month") and f.suffix.lower() == ".xlsx"]
        survey_files = [f for f in all_files if f.name.startswith("Survey_Dump") and f.suffix.lower() == ".xlsx"]
        t3_t2_files = [f for f in all_files if f.name.startswith("T3_T2_CNX") and f.suffix.lower() == ".xlsx"]

        # --- PART 5: PROCESS NEW LOOK DATA ---
        print("\n--- Processing New Look Data ---")
        new_look_dfs = []
        LOCATION_EXCEPTIONS = ["Foundever (San Salvador)", "Teleperformance (Lisbon)"]

        for file in new_look_files:
            try:
                # [QUAN TRỌNG] infer_schema_length=0: Đọc tất cả là String để tránh lỗi Schema
                df = pl.read_csv(
                    file, 
                    infer_schema_length=0, 
                    ignore_errors=True, 
                    truncate_ragged_lines=True, 
                    try_parse_dates=False
                )
                
                if "Agent Queue Group Name" in df.columns:
                    df = df.with_columns(pl.col("Agent Queue Group Name").str.strip_chars())
                if "Agent Email ID" in df.columns:
                    df = df.with_columns([
                        pl.col("Agent Email ID").str.strip_chars(), 
                        pl.col("Agent Email ID").str.strip_chars().str.to_lowercase().alias("_join_key")
                    ])
                
                # DATE FIX
                if "Joined Date" in df.columns:
                    df = df.with_columns(
                        pl.col("Joined Date")
                        .str.strip_chars()
                        .pipe(lambda c: pl.coalesce([
                            c.str.to_date(format="%m/%d/%Y", strict=False), 
                            c.str.to_date(format="%Y-%m-%d", strict=False), 
                            c.str.to_date(format="%d/%m/%Y", strict=False)
                        ]))
                        .alias("Joined date")
                    )

                # LOCATION FIX
                if "Agent Business Location" in df.columns:
                    df = df.with_columns(
                        pl.when(pl.col("Agent Business Location").is_in(LOCATION_EXCEPTIONS))
                        .then(pl.col("Agent Business Location"))
                        .otherwise(pl.col("Agent Business Location").str.extract(r"\((.*?)\)", 1))
                        .str.strip_chars()
                        .alias("Location_extracted")
                    )
                else:
                    df = df.with_columns(pl.lit(None).cast(pl.Utf8).alias("Location_extracted"))

                new_look_dfs.append(df)
            except Exception as e:
                print(f"Error reading {file.name}: {e}")

        if new_look_dfs:
            # Bây giờ concat sẽ an toàn vì tất cả đều là String
            df_new_look = pl.concat(new_look_dfs, how="diagonal").rechunk()
            
            # Joins
            if "Agent Queue Group Name" in df_new_look.columns: df_new_look = df_new_look.join(df_mapping, on="Agent Queue Group Name", how="left")
            if "_join_key" in df_new_look.columns:
                df_new_look = df_new_look.join(df_magdoom_hc, left_on="_join_key", right_on="SSO ID", how="left")
                df_new_look = df_new_look.join(df_global_hc, left_on="_join_key", right_on="SSO ID", how="left")

                cols_to_merge_standard = ["Name", "Supervisor", "AON Status"]
                merge_exprs = [pl.col(f"{col}_md").replace("0", None).replace("", None).fill_null(pl.col(f"{col}_gl")).alias(col) for col in cols_to_merge_standard]
                merge_exprs.append(pl.col("Location_extracted").replace("0", None).replace("", None).fill_null(pl.col("Location_md")).fill_null(pl.col("Location_gl")).alias("Location"))
                df_new_look = df_new_look.with_columns(merge_exprs)

            # Calculations
            outbound_expr = pl.lit(0)
            if "Initiated Outbound (Yes / No)" in df_new_look.columns: 
                outbound_expr = pl.when(pl.col("Initiated Outbound (Yes / No)") == "Yes").then(1).otherwise(0)
            
            followup_expr = pl.lit(0)
            if "Has Followup Time" in df_new_look.columns: 
                # [FIX] Xử lý dấu phẩy trước khi chuyển sang số để tránh lỗi mất dữ liệu
                followup_expr = pl.when(
                    pl.col("Has Followup Time")
                    .str.replace_all(",", "")
                    .cast(pl.Float64, strict=False)
                    .fill_null(0) > 0
                ).then(1).otherwise(0)

            def get_percentage_rate(col_name):
                return (pl.col(col_name).str.replace("%", "").cast(pl.Float64, strict=False).fill_null(0) / 100) if col_name in df_new_look.columns else pl.lit(0)

            # [FIX] Xử lý dấu phẩy cho Survey Count
            survey_col = pl.lit(0)
            if "Survey Submitted (Count)" in df_new_look.columns:
                survey_col = pl.col("Survey Submitted (Count)").str.replace_all(",", "").cast(pl.Float64, strict=False).fill_null(0)
            
            promoter_rate = get_percentage_rate("Promoter Score (Calc)")
            detractor_rate = get_percentage_rate("Detractor Score (Calc)")
            
            # Neutral với clip(0) chặn số âm
            neutral_count_expr = (
                survey_col - 
                (promoter_rate * survey_col).round(0) - 
                (detractor_rate * survey_col).round(0)
            ).round(0).clip(0)

            df_new_look = df_new_look.with_columns([
                outbound_expr.alias("Outbound"), 
                followup_expr.alias("FollowUp"),
                (promoter_rate * survey_col).round(0).alias("Promoter"), 
                (detractor_rate * survey_col).round(0).alias("Detractor"),
                neutral_count_expr.alias("Neutral"), 
                survey_col.alias("Survey")
            ])
            
            # Select Columns
            rename_map = {}
            for target in TARGET_COLUMNS:
                if target in df_new_look.columns: continue
                if target == "Joined date" and "Joined Date" in df_new_look.columns: rename_map["Joined Date"] = "Joined date"
                elif target.startswith("Conversation "):
                    short_name = target.replace("Conversation ", "", 1) 
                    if short_name in df_new_look.columns: rename_map[short_name] = target
            if rename_map: df_new_look = df_new_look.rename(rename_map)
            
            df_new_look = df_new_look.select([pl.col(col) if col in df_new_look.columns else pl.lit(None).alias(col) for col in TARGET_COLUMNS])

            # --- CAST TO NUMERIC (Handling thousands separators) ---
            print("Converting columns to numeric type (Handling thousands separators)...")
            numeric_cols = [
                "Handle (Count)", "Handle Time (Sum)", "Response Count", "Response Time", 
                "Hold Time (Sum)", "Wrap Up Time (Sum)", "Talk Time (Sum)", 
                "Outbound", "FollowUp", "Survey", "Promoter", "Detractor", "Neutral"
            ]
            
            final_numeric_exprs = []
            for col in numeric_cols:
                if col in df_new_look.columns:
                    final_numeric_exprs.append(
                        pl.col(col)
                        .cast(pl.Utf8, strict=False)    # Đảm bảo là string
                        .str.replace_all(",", "")       # Xóa dấu phẩy
                        .cast(pl.Float64, strict=False) # Chuyển sang số
                        .alias(col)
                    )
            
            if final_numeric_exprs: 
                df_new_look = df_new_look.with_columns(final_numeric_exprs)

        else:
            df_new_look = pl.DataFrame()

        # --- PART 6: PROCESS SURVEY DUMP (Strict 15 Columns) ---
        print("\n--- Processing Survey Dump ---")
        df_survey_final = pl.DataFrame()
        
        if survey_files:
            try:
                df_raw = pl.read_excel(survey_files[0])
                if "Conversation Conversation Id" in df_raw.columns: df_raw = df_raw.rename({"Conversation Conversation Id": "Conversation Id"})
                
                if "Conversation Id" in df_raw.columns and "Question Category" in df_raw.columns and "Answer" in df_raw.columns:
                    
                    df_filtered = df_raw.filter(pl.col("Question Category").is_in(TARGET_5_CATEGORIES))
                    df_filtered = df_filtered.with_columns(pl.col("Answer").cast(pl.Float64, strict=False))

                    df_pivot = df_filtered.pivot(
                        values="Answer",
                        index="Conversation Id",
                        columns="Question Category",
                        aggregate_function="first"
                    )

                    final_survey_exprs = []
                    for idx, cat_name in enumerate(TARGET_5_CATEGORIES):
                        final_survey_exprs.append(pl.col("Conversation Id").alias(f"Conversation Id_{idx+1}"))
                        if cat_name in df_pivot.columns:
                            final_survey_exprs.append(pl.col(cat_name).alias(f"Answer_{idx+1}"))
                        else:
                            final_survey_exprs.append(pl.lit(None).alias(f"Answer_{idx+1}"))
                        final_survey_exprs.append(pl.lit(cat_name).alias(f"Question Category_{idx+1}"))
                        
                    df_survey_final = df_pivot.select(final_survey_exprs)
                    print(f"Survey Pivot created with {len(df_survey_final.columns)} columns.")
                else:
                    print("Missing columns in Survey file.")
            except Exception as e:
                print(f"Error processing Survey: {e}")

        # --- OTHER SHEETS ---
        df_oa = pl.DataFrame()
        if oa_files:
            try: df_oa = pl.read_excel(oa_files[0])
            except: pass
        
        df_t3_t2 = pl.DataFrame()
        if t3_t2_files:
            try: df_t3_t2 = pl.read_excel(t3_t2_files[0]).drop(pl.col(pl.Int64))
            except: pass

        print(f"\nWriting to {output_filename}...")
        with xlsxwriter.Workbook(output_path) as workbook:
            def write_sheet(df, sheet_name):
                if not df.is_empty():
                    # Export mm/dd/yyyy
                    df.write_excel(workbook=workbook, worksheet=sheet_name, column_formats={"Joined date": "mm/dd/yyyy"})
                else:
                    workbook.add_worksheet(sheet_name).write(0, 0, "No Data Found")

            write_sheet(df_new_look, "New_Look_Data")
            write_sheet(df_oa, "OA")
            
            if not df_survey_final.is_empty():
                ws = workbook.add_worksheet("Survey_Processed")
                df_survey_final.write_excel(workbook=workbook, worksheet="Survey_Processed")
                clean_headers = []
                for _ in range(5): clean_headers.extend(["Conversation Id", "Answer", "Question Category"])
                for idx, val in enumerate(clean_headers): ws.write(0, idx, val)
            else:
                workbook.add_worksheet("Survey_Processed").write(0, 0, "No Data")
                
            write_sheet(df_t3_t2, "T3_T2")
            
        print("Done! File exported successfully.")

    except Exception as e:
        print(f"Critical Error: {e}")
        import traceback
        traceback.print_exc()

# Run
if __name__ == "__main__":
    process_and_export_final("NON_EN_Magdoom", "Consolidated_Magdoom_Report.xlsx")

Base Path: C:/Users/huuchinh.nguyen
Loading Mapping, Magdoom HC & Global HC files...


Could not determine dtype for column 6, falling back to string
Could not determine dtype for column 7, falling back to string
Could not determine dtype for column 65, falling back to string



--- Processing New Look Data ---
Converting columns to numeric type (Handling thousands separators)...

--- Processing Survey Dump ---


C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_18528\2593610891.py:227: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  df_pivot = df_filtered.pivot(


Survey Pivot created with 15 columns.


Could not determine dtype for column 10, falling back to string



Writing to Consolidated_Magdoom_Report.xlsx...
Done! File exported successfully.
